# 06  -  Deep Learning Training

**Goal:** Train one deep learning model (GRU as a representative example) through the full training pipeline: learning-rate grid search, epoch-level training loop with weighted BCE loss, early stopping monitored by validation MCC, and loss / MCC visualisation.

**Modules used:** `src/training/trainer.py`, `src/training/losses.py`, `src/training/callbacks.py`

---

## 0  -  Imports & data pipeline

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src.data.load_data import load_market_data
from src.data.preprocess import apply_missing_value_policy, select_columns
from src.data.labeling import compute_forward_return, compute_threshold, make_labels
from src.data.splitters import split_dev_test, make_expanding_folds
from src.models.model_factory import build_model
from src.training.losses import build_bce_with_logits_loss
from src.training.callbacks import EarlyStopping
from src.training.trainer import (
    create_torch_dataloader,
    predict_proba_torch_model,
    evaluate_torch_model,
    run_single_torch_fold,
)
from src.run_experiment import (
    build_labeled_sequences_for_endpoints,
    run_cv_for_model_and_scaler,
    scale_sequence_data,
)
from src.evaluation.metrics import summarize_fold_metrics
from src.utils.config import load_config
from src.utils.seed import set_global_seed

set_global_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

CONFIG_PATH = ROOT / 'configs' / 'base.yaml'
BASE_CONFIG = load_config(str(CONFIG_PATH))
DATA_PATH = ROOT / BASE_CONFIG['data']['file_path']
FEATURE_COLS = BASE_CONFIG['data']['feature_cols']
DATE_COL = BASE_CONFIG['data']['date_col']
CLOSE_COL = BASE_CONFIG['data']['close_col']
LOOKBACK = BASE_CONFIG['sequence']['lookback']
N_FOLDS = BASE_CONFIG['split']['n_folds']
VAL_RATIO = BASE_CONFIG['split']['val_ratio_within_dev']
TEST_RATIO = BASE_CONFIG['split']['final_test_ratio']


In [ ]:
df_raw   = load_market_data(str(DATA_PATH), date_col=DATE_COL)
df_clean = apply_missing_value_policy(df_raw, method='ffill_then_drop_head')
df       = select_columns(df_clean, FEATURE_COLS, CLOSE_COL, DATE_COL)

dev_df, test_df = split_dev_test(df, test_ratio=TEST_RATIO)
folds = make_expanding_folds(len(dev_df), n_folds=N_FOLDS, val_ratio_within_dev=VAL_RATIO)
print(f'Dev: {len(dev_df)} | Test: {len(test_df)} | Folds: {len(folds)}')

---
## 1  -  Prepare one fold for detailed walkthrough

We use **Fold 1** (smallest training set, fastest to run) to illustrate the full training process.

In [ ]:
tr_range, vl_range = folds[0]
train_df = dev_df.iloc[list(tr_range)].reset_index(drop=True)

train_fwd = compute_forward_return(train_df[CLOSE_COL], horizon=1)
threshold = compute_threshold(train_fwd, method='quantile', quantile=BASE_CONFIG['labeling']['threshold_quantile'])

X_tr_raw, y_tr, ts_tr, train_summary, train_diag = build_labeled_sequences_for_endpoints(
    context_df=dev_df,
    feature_cols=FEATURE_COLS,
    close_col=CLOSE_COL,
    endpoint_indices=tr_range,
    threshold=threshold,
    lookback=LOOKBACK,
    horizon=1,
    bull_label=1,
    bear_label=0,
    neutral_label=-1,
)

X_vl_raw, y_vl, ts_vl, val_summary, val_diag = build_labeled_sequences_for_endpoints(
    context_df=dev_df,
    feature_cols=FEATURE_COLS,
    close_col=CLOSE_COL,
    endpoint_indices=vl_range,
    threshold=threshold,
    lookback=LOOKBACK,
    horizon=1,
    bull_label=1,
    bear_label=0,
    neutral_label=-1,
)

X_tr, X_vl, scaler = scale_sequence_data(X_tr_raw, X_vl_raw, scaler_name='standard')

print(f'Train: X={X_tr.shape}  y={y_tr.shape}  (Bull: {(y_tr==1).sum()}, Bear: {(y_tr==0).sum()})')
print(f'Val  : X={X_vl.shape}  y={y_vl.shape}  (Bull: {(y_vl==1).sum()}, Bear: {(y_vl==0).sum()})')
print(f'Validation first endpoint uses prior train history: {val_diag["endpoint_indices"][0] - LOOKBACK + 1 < vl_range.start}')


---
## 2  -  Loss function: Weighted Binary Cross-Entropy

Because Bull and Bear classes are not perfectly balanced, we weight the positive class:

$$\text{pos\_weight} = \frac{N_{\text{bear}}}{N_{\text{bull}}}$$

This tells the loss to penalise missing Bull signals more heavily than Bear signals, balancing effective learning.

In [ ]:
loss_fn = build_bce_with_logits_loss(y_tr, use_weighted_loss=True, device=DEVICE)
print(loss_fn)

n_bear = (y_tr == 0).sum()
n_bull = (y_tr == 1).sum()
print(f'Bear samples: {n_bear}  |  Bull samples: {n_bull}')
print(f'pos_weight = {n_bear / n_bull:.4f}')

---
## 3  -  Early stopping

`EarlyStopping` monitors validation MCC each epoch. If the score does not improve for `patience=10` consecutive epochs, training halts and the best weights are restored.

In [ ]:
early_stopper = EarlyStopping(patience=10, min_delta=0.0, restore_best_state=True)
print(f'Patience         : {early_stopper.patience}')
print('Monitor metric   : validation MCC (higher is better)')
print('Behavior on stop : restore best weights')


---
## 4  -  Manual training loop (GRU  -  Fold 1)

This cell demonstrates **exactly what `run_single_torch_fold` does internally**, step by step.

In [ ]:
GRU_CONFIG = {
    'gru': {'hidden_dim': 64, 'num_layers': 1, 'dropout': 0.2, 'weighted_loss': True},
    'training': {'batch_size': 64, 'max_epochs': 60, 'learning_rate': 0.001,
                 'early_stopping_patience': 10, 'shuffle_train': True},
}

INPUT_SHAPE = (LOOKBACK, len(FEATURE_COLS))
model = build_model('gru', GRU_CONFIG, input_shape=INPUT_SHAPE).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=GRU_CONFIG['training']['learning_rate'])
early_stop = EarlyStopping(patience=GRU_CONFIG['training']['early_stopping_patience'])

train_loader = create_torch_dataloader(X_tr, y_tr, batch_size=64, shuffle=True)

train_losses, val_mccs = [], []
MAX_EPOCHS = GRU_CONFIG['training']['max_epochs']

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.float().to(DEVICE)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    val_result = evaluate_torch_model(model, X_vl, y_vl, batch_size=64, device=DEVICE)
    val_mcc = val_result['metrics']['mcc']
    val_mccs.append(val_mcc)

    improved = early_stop.step(val_mcc, model, epoch)
    marker = ' <- best' if improved else ''

    if epoch % 5 == 0 or improved:
        print(f'Epoch {epoch:3d}/{MAX_EPOCHS} | loss={avg_loss:.4f} | val_mcc={val_mcc:.4f}{marker}')

    if early_stop.should_stop:
        print(f'\nEarly stopping triggered at epoch {epoch}. Best epoch: {early_stop.best_epoch}')
        early_stop.restore(model)
        break

print('\nTraining complete.')


---
## 5  -  Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Loss curve
axes[0].plot(range(1, len(train_losses)+1), train_losses, color='steelblue', linewidth=1.2)
axes[0].set_title('Training Loss (BCE)', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

# MCC curve
best_epoch = early_stop.best_epoch
axes[1].plot(range(1, len(val_mccs)+1), val_mccs, color='mediumseagreen', linewidth=1.2)
axes[1].axvline(best_epoch, color='red', linestyle='--', linewidth=1, label=f'Best epoch ({best_epoch})')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle=':')
axes[1].set_title('Validation MCC', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MCC')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6 - Full CV using `run_cv_for_model_and_scaler`

The high-level helper below is the same path used by `run_experiment.py`. It adds LR search, endpoint-safe sequence construction, internal validation for probability-threshold selection, and fold-level diagnostics.


In [ ]:
FULL_CONFIG = load_config(str(CONFIG_PATH))
FULL_CONFIG['models']['enabled'] = ['gru']
FULL_CONFIG['preprocessing']['scalers'] = ['standard']

result = run_cv_for_model_and_scaler(
    config=FULL_CONFIG,
    dev_df=dev_df,
    model_name='gru',
    scaler_name='standard',
)

summary_dl = result['cv_summary']
print(f'GRU CV - MCC: {summary_dl["mcc_mean"]:.4f} +/- {summary_dl["mcc_std"]:.4f}')
print(f'GRU CV - F1 : {summary_dl["f1_mean"]:.4f} +/- {summary_dl["f1_std"]:.4f}')
print(f'GRU CV - PR-AUC: {(summary_dl.get("pr_auc_mean") or 0):.4f}')

fold_thresholds = [fold['selected_probability_threshold'] for fold in result['fold_results']]
print('Selected probability thresholds by fold:', fold_thresholds)


## Summary

| Component | Implementation |
|---|---|
| Loss function | `BCEWithLogitsLoss` + `pos_weight` for class imbalance |
| Optimizer | Adam |
| Early stopping | Patience-based validation MCC, restore best weights |
| Train loader | `shuffle_train` can be enabled because each sample is already past-only |
| CV | Expanding window, endpoint-safe sequences, scaler fit on train only |
| Decision threshold | Selected on internal validation data, never on final test |

**Next:** `07_evaluation_and_comparison.ipynb` - confusion matrices, baselines, PR-AUC, and best-model selection.
